# RecSys: Item-to-Item (Co-occurrence) + Context Fallback



## What does?

Builds an item-to-item recommender using **order-level co-occurrence** on TRAIN only (temporal split),
stores ONLY the "real neighbors" into Postgres, and keeps fallback logic for serving time.

## Approach

Olist has very few multi-item orders. If we try to precompute (materialize) recommendations for the entire catalog
(33k items) we would upsert tens of thousands of large JSON rows, which is slow and mostly redundant because
fallback recommendations already exist in `recommendations_context`

stores only items that have co-occurrence evidence.
At serving time:
  - if item neighbors exist -> return them
  - else fallback to category/global popularity from `recommendations_context`

## DB assumptions

Tables:
- interactions(user_id, item_id, ts, event_type, order_id)
- items(item_id, category_en)
- recommendations_context(context_type, context_value, model, recs, generated_at)
- recommendations_items(item_id, model, recs, generated_at)   <-- already in your init.sql


# Imports and DB helper

In [24]:
import json
from itertools import combinations

import numpy as np
import pandas as pd
import psycopg


from src.db.conn import get_db_conninfo

conninfo = get_db_conninfo()

# Temporal cutoff 

In [25]:
CUTOFF_QUANTILE = 0.80

with psycopg.connect(conninfo) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT percentile_disc(%s) WITHIN GROUP (ORDER BY ts) AS cutoff_ts
            FROM interactions
            WHERE event_type='purchase';
            """,
            (CUTOFF_QUANTILE,),
        )
        cutoff_ts = cur.fetchone()[0]

print("cutoff_ts:", cutoff_ts)


cutoff_ts: 2018-05-26 19:36:13+00:00


# Measure basket analysis

In [26]:
with psycopg.connect(conninfo) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT order_id::text, item_id::text
            FROM interactions
            WHERE event_type='purchase'
              AND ts < %s
              AND order_id IS NOT NULL;
            """,
            (cutoff_ts,),
        )
        rows = cur.fetchall()

df_train_orders = pd.DataFrame.from_records(rows, columns=["order_id", "item_id"])

# Quick basket sparsity summary
order_sizes = df_train_orders.groupby("order_id")["item_id"].nunique()
summary = pd.Series(
    {
        "n_orders_total": float(order_sizes.shape[0]),
        "n_orders_1_item": float((order_sizes == 1).sum()),
        "n_orders_2plus": float((order_sizes >= 2).sum()),
        "pct_orders_1_item": float((order_sizes == 1).mean()),
        "pct_orders_2plus": float((order_sizes >= 2).mean()),
        "avg_items_per_order": float(order_sizes.mean()),
    }
)
print(summary)

n_orders_total         77173.000000
n_orders_1_item        74613.000000
n_orders_2plus          2560.000000
pct_orders_1_item          0.966828
pct_orders_2plus           0.033172
avg_items_per_order        1.038329
dtype: float64


# Load item -> category mapping (sanity checks)

In [27]:
with psycopg.connect(conninfo) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT item_id::text, category_en::text FROM items;")
        items_rows = cur.fetchall()

items_cat = pd.DataFrame(items_rows, columns=["item_id", "category_en"])
items_cat["category_en"] = items_cat["category_en"].astype("string").fillna("unknown")
item2cat = dict(
    zip(items_cat["item_id"].astype(str), items_cat["category_en"].astype(str))
)

# Build co-occurrence pairs from multi-item orders

In [28]:
order_items = df_train_orders.groupby("order_id")["item_id"].apply(
    lambda x: list(pd.unique(x))
)
order_items = order_items[order_items.apply(len) >= 2]
print("orders with 2+ items:", len(order_items))

# frequency of items within multi-item orders only
item_freq = pd.Series([it for lst in order_items for it in lst]).value_counts()

pair_counts = {}
for lst in order_items:
    lst = sorted(lst)
    for a, b in combinations(lst, 2):
        pair_counts[(a, b)] = pair_counts.get((a, b), 0) + 1

pairs = pd.DataFrame(
    [(a, b, c) for (a, b), c in pair_counts.items()],
    columns=["item_a", "item_b", "cooc"],
)

pairs["freq_a"] = pairs["item_a"].map(item_freq)
pairs["freq_b"] = pairs["item_b"].map(item_freq)
pairs["score"] = pairs["cooc"] / np.sqrt(pairs["freq_a"] * pairs["freq_b"])

print("pairs total:", len(pairs))


orders with 2+ items: 2560
pairs total: 3135


# Build neighbors 

In [29]:
TOPK = 30
MIN_COOC = 2  # 2 for higher precision but low coverage
MODEL_NAME = f"cooc_cosine_mincooc{MIN_COOC}_top{TOPK}"

pairs_f = pairs[pairs["cooc"] >= MIN_COOC].copy()
print("pairs after MIN_COOC:", len(pairs_f))

edges = pd.concat(
    [
        pairs_f.rename(columns={"item_a": "item_id", "item_b": "neighbor_item_id"})[
            ["item_id", "neighbor_item_id", "score", "cooc"]
        ],
        pairs_f.rename(columns={"item_b": "item_id", "item_a": "neighbor_item_id"})[
            ["item_id", "neighbor_item_id", "score", "cooc"]
        ],
    ],
    ignore_index=True,
)

edges = edges.sort_values(["item_id", "score", "cooc"], ascending=[True, False, False])
edges["rank"] = edges.groupby("item_id").cumcount() + 1
neighbors_cooc = edges[edges["rank"] <= TOPK].copy()

print("neighbors rows:", len(neighbors_cooc))
print(f"items with >={MIN_COOC} neighbor:", neighbors_cooc["item_id"].nunique())

# Catalog coverage (informational)
n_catalog = items_cat["item_id"].nunique()
coverage = neighbors_cooc["item_id"].nunique() / max(n_catalog, 1)
print("catalog items:", n_catalog, "| coverage (has neighbors):", coverage)

pairs after MIN_COOC: 162
neighbors rows: 324
items with >=2 neighbor: 244
catalog items: 32951 | coverage (has neighbors): 0.0074049345998603985


# Same category check

In [30]:
nb = neighbors_cooc.merge(items_cat, on="item_id", how="left").rename(
    columns={"category_en": "cat_item"}
)
nb = nb.merge(
    items_cat,
    left_on="neighbor_item_id",
    right_on="item_id",
    how="left",
    suffixes=("", "_nb"),
)
nb = nb.rename(columns={"category_en": "cat_nb"}).drop(columns=["item_id_nb"])

top10 = nb[nb["rank"] <= 10].copy()
same_cat_rate = float((top10["cat_item"] == top10["cat_nb"]).mean())
print("same_cat_rate (top10):", same_cat_rate)

same_cat_rate (top10): 0.9444444444444444


# Store only neighbors

In [31]:
# Prepare rows for upsert: only items that actually have neighbors in neighbors_cooc
rows_to_upsert = []
for item_id, grp in neighbors_cooc.groupby("item_id"):
    grp = grp.sort_values("rank")
    recs = [
        {"item_id": str(nb), "score": float(sc), "cooc": int(co)}
        for nb, sc, co in zip(grp["neighbor_item_id"], grp["score"], grp["cooc"])
    ]
    rows_to_upsert.append((str(item_id), MODEL_NAME, json.dumps(recs)))

print("Prepared rows for upsert (real neighbors only):", len(rows_to_upsert))


def ensure_table_recommendations_items(conn):
    sql = """
    CREATE TABLE IF NOT EXISTS recommendations_items (
      item_id      text NOT NULL,
      model        text NOT NULL,
      recs         jsonb NOT NULL,
      generated_at timestamptz NOT NULL DEFAULT now(),
      PRIMARY KEY (item_id, model)
    );
    """
    with conn.cursor() as cur:
        cur.execute(sql)
    conn.commit()


def upsert_item_recs_chunked(conn, rows, batch_size=500):
    sql = """
    INSERT INTO recommendations_items(item_id, model, recs, generated_at)
    VALUES (%s, %s, %s::jsonb, now())
    ON CONFLICT (item_id, model)
    DO UPDATE SET recs = EXCLUDED.recs,
                  generated_at = EXCLUDED.generated_at;
    """
    with conn.cursor() as cur:
        for start in range(0, len(rows), batch_size):
            chunk = rows[start : start + batch_size]
            cur.executemany(sql, chunk)
            conn.commit()
            print(f"Upserted {min(start + batch_size, len(rows))}/{len(rows)}")


with psycopg.connect(conninfo) as conn:
    ensure_table_recommendations_items(conn)

    upsert_item_recs_chunked(conn, rows_to_upsert, batch_size=500)

print("OK: recommendations_items updated. model =", MODEL_NAME)

Prepared rows for upsert (real neighbors only): 244
Upserted 244/244
OK: recommendations_items updated. model = cooc_cosine_mincooc2_top30


# Show one row

In [32]:
# Pick an item that is guaranteed to have neighbors
sample_item = neighbors_cooc["item_id"].iloc[0]
sample_cat = item2cat.get(sample_item, "unknown")

print("sample_item:", sample_item, "| category:", sample_cat)

stored = None
with psycopg.connect(conninfo) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT recs
            FROM recommendations_items
            WHERE item_id=%s AND model=%s
            LIMIT 1;
            """,
            (sample_item, MODEL_NAME),
        )
        row = cur.fetchone()
        stored = row[0] if row else None

# recommendations must be python list (jsonb)
if isinstance(stored, str):
    stored = json.loads(stored)

print("stored (cooc) recs (first 5):", stored[:5] if stored else None)

sample_item: 005030ef108f58b46b78116f754d8d38 | category: perfumery
stored (cooc) recs (first 5): [{'cooc': 2, 'score': 0.6666666666666666, 'item_id': '51250f90d798d377a1928e8a4e2e9ae1'}, {'cooc': 2, 'score': 0.6666666666666666, 'item_id': '75c06ee06b201f9b6301d2b5e72993f8'}]


In [33]:
from src.etl.build_item_item_cooccurrence import run

out = run(write=False)
out["same_cat_rate_top10"], out["neighbors_meta"]

(0.9444444444444444,
 {'pairs_after_min_cooc': 162,
  'neighbors_rows': 324,
  'items_with_neighbors': 244,
  'catalog_items': 32951,
  'coverage': 0.0074049345998603985})